# Spindle: End-to-End Walkthrough

This notebook walks through the core functionality of **Spindle** — an LLM-powered, ontology-first knowledge graph extraction toolkit. You'll learn how to:

1. Configure Spindle and define an ontology
2. Extract structured triples from unstructured text
3. Store and query triples in a graph database
4. Use vector search for semantic retrieval
5. Run entity resolution to deduplicate your graph
6. Work with the Knowledge Organization System (KOS)
7. Set up spindle-eval for pipeline evaluation

**Prerequisites:**
- `uv pip install -e ".[dev]"` from the repo root
- An `ANTHROPIC_API_KEY` environment variable (for LLM-powered extraction)

All stores created in this notebook use a temporary directory and are cleaned up at the end.

---
## 1. Setup & Configuration

Spindle organises all persistent state (graphs, vectors, KOS data) under a single **stores root** directory. `SpindleConfig` manages paths and LLM credentials. For this notebook we'll use a temporary directory so nothing persists after the session.

In [ ]:
import os
import tempfile
from pathlib import Path

from dotenv import load_dotenv

from spindle.configuration import SpindleConfig

# Load environment variables from .env (if present)
load_dotenv()

# Create a temporary stores root for this session
tmp_dir = tempfile.mkdtemp(prefix="spindle_walkthrough_")
stores_root = Path(tmp_dir)

config = SpindleConfig.with_auto_detected_llm(str(stores_root))
config.storage.ensure_directories()

print(f"Stores root: {stores_root}")
print(f"Graphs dir:  {config.storage.graphs_dir}")
print(f"Vector dir:  {config.storage.vector_store_dir}")
print(f"KOS dir:     {config.storage.kos_dir}")

keys = ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY','LANGFUSE_PUBLIC_KEY']
for key in keys:
    if os.getenv(key):
        print(f"{key} detected.")
    else:
        print(f"{key} not found.")

/Users/danielwood/Repos/spindle/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Stores root: /var/folders/7q/2b44gz516yn4bt_1_4x0kmf40000gn/T/spindle_walkthrough_u4hl8f4r
Graphs dir:  /var/folders/7q/2b44gz516yn4bt_1_4x0kmf40000gn/T/spindle_walkthrough_u4hl8f4r/graphs
Vector dir:  /var/folders/7q/2b44gz516yn4bt_1_4x0kmf40000gn/T/spindle_walkthrough_u4hl8f4r/vector_store
KOS dir:     /var/folders/7q/2b44gz516yn4bt_1_4x0kmf40000gn/T/spindle_walkthrough_u4hl8f4r/kos
ANTHROPIC_API_KEY detected.
LANGFUSE_PUBLIC_KEY detected.


---
## 2. Define an Ontology

Spindle is **ontology-first** — you define the entity types and relation types you care about, and the LLM extracts only what fits that schema. This gives you structured, predictable output instead of open-ended NER.

We'll model a small research publications domain: researchers, institutions, papers, and topics.

In [2]:
from spindle import create_ontology

entity_types = [
    {
        "name": "Researcher",
        "description": "A person who conducts scientific research",
        "attributes": [
            {"name": "title", "type": "string", "description": "Academic title (e.g. Professor, Dr.)"},
        ],
    },
    {
        "name": "Institution",
        "description": "A university, lab, or research organisation",
    },
    {
        "name": "Paper",
        "description": "A published research paper or article",
        "attributes": [
            {"name": "year", "type": "int", "description": "Year of publication"},
        ],
    },
    {
        "name": "Topic",
        "description": "A research topic or field of study",
    },
]

relation_types = [
    {"name": "affiliated_with", "description": "Researcher is affiliated with an institution", "domain": "Researcher", "range": "Institution"},
    {"name": "authored", "description": "Researcher authored a paper", "domain": "Researcher", "range": "Paper"},
    {"name": "covers_topic", "description": "Paper covers a research topic", "domain": "Paper", "range": "Topic"},
    {"name": "collaborates_with", "description": "Two researchers collaborate", "domain": "Researcher", "range": "Researcher"},
]

ontology = create_ontology(entity_types, relation_types)

print(f"Entity types: {[et.name for et in ontology.entity_types]}")
print(f"Relation types: {[rt.name for rt in ontology.relation_types]}")

Entity types: ['Researcher', 'Institution', 'Paper', 'Topic']
Relation types: ['affiliated_with', 'authored', 'covers_topic', 'collaborates_with']


You can also inspect the ontology as a dictionary, which is useful for serialisation or debugging.

In [3]:
from spindle import ontology_to_dict

ontology_to_dict(ontology)

{'entity_types': [{'name': 'Researcher',
   'description': 'A person who conducts scientific research',
   'attributes': [{'name': 'title',
     'type': 'string',
     'description': 'Academic title (e.g. Professor, Dr.)'}]},
  {'name': 'Institution',
   'description': 'A university, lab, or research organisation',
   'attributes': []},
  {'name': 'Paper',
   'description': 'A published research paper or article',
   'attributes': [{'name': 'year',
     'type': 'int',
     'description': 'Year of publication'}]},
  {'name': 'Topic',
   'description': 'A research topic or field of study',
   'attributes': []}],
 'relation_types': [{'name': 'affiliated_with',
   'description': 'Researcher is affiliated with an institution',
   'domain': 'Researcher',
   'range': 'Institution'},
  {'name': 'authored',
   'description': 'Researcher authored a paper',
   'domain': 'Researcher',
   'range': 'Paper'},
  {'name': 'covers_topic',
   'description': 'Paper covers a research topic',
   'domain': '

---
## 3. Extract Triples from Text

`SpindleExtractor` sends text to an LLM along with your ontology schema. The LLM returns structured triples that conform to your entity and relation types. Each triple includes the subject, predicate, object, source provenance, and the supporting text spans from the original document.

> **Note:** This section calls the Anthropic API and requires a valid `ANTHROPIC_API_KEY`.

In [4]:
import os

from spindle import SpindleExtractor

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Add it to your environment or .env, "
        "re-run setup cell 1, then run this cell again."
    )

extractor = SpindleExtractor(ontology=ontology)

text_1 = """
Professor Maria Chen at MIT published a landmark paper titled "Scaling Laws for
Neural Language Models" in 2023. The paper explores how model performance changes
with increasing parameter counts and training data. Dr. Chen collaborated with
James Liu from Stanford University on this work, which covers topics in deep
learning and natural language processing.
""".strip()

result_1 = extractor.extract(text_1, source_name="article_1", source_url="https://example.com/article1")

print(f"Extracted {len(result_1.triples)} triples")
print(f"\nReasoning: {result_1.reasoning[:200]}...")

2026-03-13T13:58:37.694 [BAML INFO] Function ExtractTriples:
    Client: CustomHaiku (claude-haiku-4-5-20251001) - 10205ms. StopReason: end_turn. Tokens(in/out): 1599/1890
    ---PROMPT---
    user: You are a knowledge graph extraction expert. Your task is to extract structured triples (subject-predicate-object) from the provided text, with rich entity metadata, custom attributes, and supporting evidence.ONTOLOGY:
    You must extract triples that conform to the following ontology:
    
    Valid Entity Types:
    - Researcher: A person who conducts scientific research
      Custom Attributes:
        * title (string): Academic title (e.g. Professor, Dr.)
    - Institution: A university, lab, or research organisation
    - Paper: A published research paper or article
      Custom Attributes:
        * year (int): Year of publication
    - Topic: A research topic or field of study
    
    Valid Relation Types:
    - affiliated_with: Researcher is affiliated with an institution
      (D

Transient error Internal Server Error encountered while exporting span batch, retrying in 0.91s.
Transient error Internal Server Error encountered while exporting span batch, retrying in 2.09s.
Failed to export span batch due to timeout, max retries or shutdown.


In [5]:
# Inspect each extracted triple
for i, triple in enumerate(result_1.triples):
    print(f"\n--- Triple {i+1} ---")
    print(f"  {triple.subject.name} ({triple.subject.type})")
    print(f"    --[{triple.predicate}]-->")
    print(f"  {triple.object.name} ({triple.object.type})")
    if triple.subject.custom_atts:
        print(f"  Subject attributes: {triple.subject.custom_atts}")
    if triple.supporting_spans:
        print(f"  Supporting text: \"{triple.supporting_spans[0].text[:80]}...\"")


--- Triple 1 ---
  Maria Chen (Researcher)
    --[affiliated_with]-->
  MIT (Institution)
  Subject attributes: {'title': AttributeValue(value='Professor', type='string')}
  Supporting text: "Professor Maria Chen at MIT..."

--- Triple 2 ---
  Maria Chen (Researcher)
    --[authored]-->
  Scaling Laws for Neural Language Models (Paper)
  Subject attributes: {'title': AttributeValue(value='Professor', type='string')}
  Supporting text: "Professor Maria Chen at MIT published a landmark paper titled "Scaling Laws for
..."

--- Triple 3 ---
  James Liu (Researcher)
    --[affiliated_with]-->
  Stanford University (Institution)
  Subject attributes: {'title': AttributeValue(value='Dr.', type='string')}
  Supporting text: "James Liu from Stanford University..."

--- Triple 4 ---
  James Liu (Researcher)
    --[authored]-->
  Scaling Laws for Neural Language Models (Paper)
  Subject attributes: {'title': AttributeValue(value='Dr.', type='string')}
  Supporting text: "Dr. Chen collaborated wi

Let's extract from a second passage to build up a richer graph. Notice we pass `existing_triples` so the LLM can maintain entity consistency across extractions.

In [6]:
text_2 = """
Dr. James Liu, a researcher at Stanford, recently published "Efficient Fine-Tuning
Methods for Large Language Models" (2024). The paper builds on prior scaling law
research and proposes novel parameter-efficient approaches. Liu's colleague Sarah
Park at Stanford co-authored the study, which advances the field of transfer learning.
""".strip()

result_2 = extractor.extract(
    text_2,
    source_name="article_2",
    existing_triples=result_1.triples,  # entity consistency
)

print(f"Extracted {len(result_2.triples)} triples from second passage")

all_triples = result_1.triples + result_2.triples
print(f"Total triples: {len(all_triples)}")

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


2026-03-13T13:47:41.609 [BAML WARN] Function ExtractTriples:
    (3 other previous tries)
    Client: CustomHaiku (<unknown>) - 104ms
    ---PROMPT---
    user: You are a knowledge graph extraction expert. Your task is to extract structured triples (subject-predicate-object) from the provided text, with rich entity metadata, custom attributes, and supporting evidence.ONTOLOGY:
    You must extract triples that conform to the following ontology:
    
    Valid Entity Types:
    - Researcher: A person who conducts scientific research
      Custom Attributes:
        * title (string): Academic title (e.g. Professor, Dr.)
    - Institution: A university, lab, or research organisation
    - Paper: A published research paper or article
      Custom Attributes:
        * year (int): Year of publication
    - Topic: A research topic or field of study
    
    Valid Relation Types:
    - affiliated_with: Researcher is affiliated with an institution
      (Domain: Researcher, Range: Institution)

BamlClientHttpError: BamlClientHttpError(client_name=CustomHaiku, message=Request failed with status code: 404 Not Found. {"type":"error","error":{"type":"not_found_error","message":"model: claude-3-5-haiku-latest"},"request_id":"req_011CZ1aUnTs1MhY5JAYh7RDi"}, status_code=404, detailed_message=4 failed attempts:

Attempt 0: LLM client "CustomHaiku" failed with status code: Unspecified error code: 404
    Message: Request failed with status code: 404 Not Found. {"type":"error","error":{"type":"not_found_error","message":"model: claude-3-5-haiku-latest"},"request_id":"req_011CZ1aUiMZJEbH9Ldr7JZRo"}
Attempt 1: LLM client "CustomHaiku" failed with status code: Unspecified error code: 404
    Message: Request failed with status code: 404 Not Found. {"type":"error","error":{"type":"not_found_error","message":"model: claude-3-5-haiku-latest"},"request_id":"req_011CZ1aUjndEpgEP6DqF1J5X"}
Attempt 2: LLM client "CustomHaiku" failed with status code: Unspecified error code: 404
    Message: Requ

---
## 4. Work with Extraction Results

Spindle provides utility functions for serialising, filtering, and inspecting triples. These are useful for exporting results, debugging extractions, or building downstream pipelines.

In [ ]:
from spindle import triples_to_dict, dict_to_triples, filter_triples_by_source, get_supporting_text

# Serialise to JSON-safe dicts
triple_dicts = triples_to_dict(all_triples)
print("First triple as dict:")
triple_dicts[0]

In [ ]:
# Roundtrip: dicts -> triples
roundtripped = dict_to_triples(triple_dicts)
print(f"Roundtripped {len(roundtripped)} triples successfully")

In [ ]:
# Filter triples by source document
article_1_triples = filter_triples_by_source(all_triples, "article_1")
article_2_triples = filter_triples_by_source(all_triples, "article_2")

print(f"Triples from article_1: {len(article_1_triples)}")
print(f"Triples from article_2: {len(article_2_triples)}")

In [ ]:
# Get the raw supporting text for each triple
for triple in all_triples[:3]:
    spans = get_supporting_text(triple)
    print(f"{triple.subject.name} --[{triple.predicate}]--> {triple.object.name}")
    for span in spans:
        print(f"  Evidence: \"{span[:80]}...\"")
    print()

---
## 5. Store Triples in a Graph

`GraphStore` persists your extracted triples in a [Kuzu](https://kuzudb.com/) graph database. This gives you a queryable knowledge graph with support for pattern matching, Cypher queries, and source/date filtering.

In [ ]:
from spindle import GraphStore

graph_store = GraphStore(db_path="walkthrough_graph", config=config)

# Add all triples at once
edges_added = graph_store.add_triples(all_triples)
print(f"Edges added: {edges_added}")

stats = graph_store.get_statistics()
print(f"Graph statistics: {stats}")

---
## 6. Query the Graph

You can query the graph by pattern (subject, predicate, object), by source document, or with raw Cypher for full flexibility.

In [ ]:
# Pattern query: find all "authored" relationships
authored = graph_store.query_by_pattern(predicate="authored")
print("Authored relationships:")
for edge in authored:
    print(f"  {edge.get('subject', '')} --> {edge.get('object', '')}")

In [ ]:
# Source query: find all triples from article_2
from_article_2 = graph_store.query_by_source("article_2")
print(f"Triples from article_2: {len(from_article_2)}")
for edge in from_article_2:
    print(f"  {edge.get('subject', '')} --[{edge.get('predicate', '')}]--> {edge.get('object', '')}")

In [ ]:
# Cypher query: find all researchers and their institutions
cypher_results = graph_store.query_cypher("""
    MATCH (s:Entity)-[r:Relationship]->(o:Entity)
    WHERE r.predicate = 'affiliated_with'
    RETURN s.name AS researcher, o.name AS institution
""")

print("Researcher affiliations (via Cypher):")
for row in cypher_results:
    print(f"  {row['researcher']} @ {row['institution']}")

In [ ]:
# List all nodes
nodes = graph_store.nodes()
print(f"\nAll {len(nodes)} nodes in graph:")
for node in nodes:
    print(f"  {node.get('name', '')} ({node.get('entity_type', '')})")

---
## 7. Vector Store & Semantic Search

`ChromaVectorStore` wraps [ChromaDB](https://www.trychroma.com/) for embedding-based semantic search. It auto-selects the best available embedding backend (local sentence-transformers preferred, then API-based). The vector store is also required for entity resolution (next section).

In [ ]:
from spindle import ChromaVectorStore

vector_store = ChromaVectorStore(
    collection_name="walkthrough_embeddings",
    persist_directory=str(config.storage.vector_store_dir),
)

# Add some text for semantic search
texts_to_embed = [
    (text_1, {"source": "article_1"}),
    (text_2, {"source": "article_2"}),
]

for text, metadata in texts_to_embed:
    uid = vector_store.add(text, metadata=metadata)
    print(f"Added document (uid={uid[:8]}...)")

In [ ]:
# Semantic search
results = vector_store.query("language model scaling", top_k=2)

print("Semantic search results for 'language model scaling':")
for r in results:
    print(f"\n  Distance: {r['distance']:.4f}")
    print(f"  Source: {r.get('metadata', {}).get('source', 'unknown')}")
    print(f"  Text: {r.get('text', '')[:100]}...")

---
## 8. Entity Resolution

When extracting from multiple documents, the same real-world entity may appear with different names (e.g. "Dr. James Liu" vs "James Liu" vs "Liu"). Entity resolution finds these duplicates and links them with `SAME_AS` edges.

The pipeline works in three stages:
1. **Semantic Blocking** — cluster entities by embedding similarity to reduce pairwise comparisons
2. **Semantic Matching** — use the LLM to confirm whether candidate pairs are truly the same entity
3. **Edge Creation** — write `SAME_AS` edges into the graph store

> **Note:** This calls the Anthropic API for LLM-based matching.

In [ ]:
from spindle import EntityResolver, ResolutionConfig, resolve_entities

resolution_config = ResolutionConfig(
    blocking_threshold=0.85,
    matching_threshold=0.8,
    clustering_method="hierarchical",
)

resolution_result = resolve_entities(
    graph_store,
    vector_store,
    config=resolution_config,
    context="Research publications about AI and machine learning",
)

print(f"Nodes processed:      {resolution_result.total_nodes_processed}")
print(f"Blocks created:       {resolution_result.blocks_created}")
print(f"Node matches found:   {len(resolution_result.node_matches)}")
print(f"SAME_AS edges added:  {resolution_result.same_as_edges_created}")
print(f"Duplicate clusters:   {resolution_result.duplicate_clusters}")
print(f"Execution time:       {resolution_result.execution_time_seconds:.2f}s")

In [ ]:
# Query with deduplication awareness
deduped = graph_store.query_with_resolution(predicate="authored")
print("Authored relationships (deduplicated):")
for edge in deduped:
    print(f"  {edge.get('subject', '')} --> {edge.get('object', '')}")

---
## 9. Knowledge Organisation System (KOS)

`KOSService` provides an in-process SKOS/OWL knowledge organisation system backed by [pyoxigraph](https://github.com/oxigraph/oxigraph). It supports concept CRUD, Aho-Corasick NER, approximate nearest-neighbour search, SPARQL queries, and SHACL validation.

The KOS is used internally by the pipeline for ontology synthesis, but you can also use it directly for managing a controlled vocabulary.

In [ ]:
from spindle import KOSService

kos = KOSService(kos_dir=str(config.storage.kos_dir))

print(f"KOS stats: {kos.stats()}")

In [ ]:
# Create concepts in the controlled vocabulary
ai = kos.create_concept("Artificial Intelligence", definition="The simulation of human intelligence by machines")
ml = kos.create_concept("Machine Learning", definition="A subset of AI focused on learning from data", broader=ai.uri)
dl = kos.create_concept("Deep Learning", definition="Neural network-based machine learning", broader=ml.uri)
nlp = kos.create_concept("Natural Language Processing", definition="AI for understanding human language", broader=ai.uri)
tl = kos.create_concept(
    "Transfer Learning",
    definition="Reusing a trained model on a new task",
    broader=ml.uri,
    alt_labels=["domain adaptation", "fine-tuning"],
)

print(f"Created concepts. KOS stats: {kos.stats()}")

In [ ]:
# Aho-Corasick NER: find known concepts in text
kos.reload()  # rebuild indices after adding concepts

sample_text = "Recent advances in deep learning and transfer learning have improved natural language processing."
mentions = kos.search_ahocorasick(sample_text)

print("NER mentions found:")
for m in mentions:
    print(f"  '{m.text}' at [{m.start}:{m.end}] -> {m.concept_uri}")

In [ ]:
# Browse the concept hierarchy
hierarchy = kos.get_hierarchy(ai.uri, depth=3)
print(f"Hierarchy from '{ai.pref_label}':")

def print_hierarchy(node, indent=0):
    print(f"{'  ' * indent}- {node.get('label', node.get('uri', '?'))}")
    for child in node.get("children", []):
        print_hierarchy(child, indent + 1)

print_hierarchy(hierarchy)

In [ ]:
# SPARQL query: find all concepts with broader relationships
sparql_results = kos.sparql("""
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?label ?broader_label
    WHERE {
        GRAPH ?g {
            ?concept skos:prefLabel ?label .
            ?concept skos:broader ?broader .
            ?broader skos:prefLabel ?broader_label .
        }
    }
""")

print("Concept hierarchy via SPARQL:")
for row in sparql_results:
    print(f"  {row.get('label', '')} -> broader -> {row.get('broader_label', '')}")

---
## 10. Setting Up spindle-eval

**spindle-eval** is a separate evaluation framework that can orchestrate Spindle's pipeline stages with metrics, gating, and experiment tracking. Spindle integrates with it via `get_pipeline_definition()`, which returns a list of `StageDef` objects that spindle-eval's runner can execute.

This section shows how to install and configure the integration. **spindle-eval is an optional dependency** — Spindle works fully standalone without it.

### Installation

```bash
# Install spindle with eval support
uv pip install -e ".[eval]"
```

### Pipeline Definition

`get_pipeline_definition()` assembles the full pipeline as a list of `StageDef` objects. Each stage declares its `input_keys` — a mapping that tells the runner how to route outputs from upstream stages as inputs to downstream ones.

```
preprocessing  (no inputs — reads documents directly)
    └──> kos_extraction  (input: preprocessing.chunks)
            └──> ontology_synthesis  (input: kos_extraction.kos_path)
                    └──> retrieval  (input: ontology_synthesis.graph)
                            └──> generation  (input: retrieval.contexts)
```

In [ ]:
from spindle import get_pipeline_definition

if get_pipeline_definition is not None:
    stages = get_pipeline_definition(
        cfg=None,                    # optional Hydra config
        ontology=ontology,           # reuse our ontology from above
        graph_store=graph_store,
        vector_store=vector_store,
    )

    print(f"Pipeline has {len(stages)} stages:\n")
    for stage in stages:
        gate_str = " (gated)" if stage.gate else ""
        metrics_str = f", {len(stage.metrics)} metrics" if stage.metrics else ""
        print(f"  {stage.name}{gate_str}{metrics_str}")
        if stage.input_keys:
            for k, v in stage.input_keys.items():
                print(f"    input: {k} <- {v}")
else:
    print("spindle-eval is not installed. Run: uv pip install -e '.[eval]'")

### Pipeline Stages

Each stage can also be used independently outside of spindle-eval. Here's a quick reference:

| Stage | What it does |
|---|---|
| `PreprocessingStage` | Document ingestion (Docling), chunking (Chonkie), coreference resolution (fastcoref) |
| `KOSExtractionStage` | Cold-start or incremental concept extraction into the KOS |
| `OntologySynthesisStage` | Generates OWL ontology + SHACL shapes from the KOS |
| `RetrievalStage` | Hybrid retrieval combining KOS, vector, and graph search |
| `GenerationStage` | LLM-powered triple extraction using `SpindleExtractor` |
| `EntityResolutionStage` | Semantic deduplication (not in the default pipeline — run as a post-step) |

In [ ]:
# Individual stages can be used directly
from spindle.stages import GenerationStage

gen_stage = GenerationStage(ontology=ontology)

print(f"Stage name:    {gen_stage.name}")
print(f"Input schema:  {gen_stage.input_schema()}")
print(f"Output schema: {gen_stage.output_schema()}")

### Hydra Configuration

Spindle ships Hydra config groups under `spindle/conf/` that spindle-eval picks up automatically via a search path plugin. You can override any stage's defaults:

```yaml
# your_eval_config.yaml
defaults:
  - preprocessing: spindle_default    # or spindle_fast (skips coref)
  - kos_extraction: cold_start        # or incremental
  - ontology_synthesis: default
  - retrieval: hybrid                 # or local, global
  - generation: default

# Override specific values
preprocessing:
  chunk_size: 800
retrieval:
  top_k: 20
```

Available config groups and their presets:

| Group | Presets | Key parameters |
|---|---|---|
| `preprocessing` | `spindle_default`, `spindle_fast` | `chunk_size`, `overlap`, `coref_model` |
| `kos_extraction` | `cold_start`, `incremental` | `mode`, `llm_model`, `max_concepts_per_chunk` |
| `ontology_synthesis` | `default` | `max_axioms_per_class`, `generate_shacl` |
| `retrieval` | `hybrid`, `local`, `global` | `strategy`, `top_k`, `vector_weight` |
| `generation` | `default` | `llm_model`, `batch_mode` |

---
## 11. Cleanup

Close the stores and remove the temporary directory. In production, you'd skip the cleanup to persist your data across sessions.

In [ ]:
import shutil

graph_store.close()
vector_store.close()

# Remove temporary stores
shutil.rmtree(tmp_dir, ignore_errors=True)
print(f"Cleaned up {tmp_dir}")

---
## Next Steps

- **Preprocessing**: Use `SpindlePreprocessor` to ingest PDFs, DOCX, and other document formats via Docling before extraction
- **Batch extraction**: Use `extractor.extract_batch()` or `extract_batch_stream()` for processing multiple documents
- **Custom embeddings**: Pass your own embedding function to `ChromaVectorStore` or use `create_openai_embedding_function()` / `create_huggingface_embedding_function()`
- **Persistence**: Point `SpindleConfig.with_root()` at a permanent directory to persist your graph and vector stores
- **API server**: Run `uv run spindle-api` to start the FastAPI REST server for extraction and resolution